# 🧩 07 - Grouping, Summarizing, and Combining Data

So far, every question has been about the **whole** dataset or **one row** at a time. Real questions are usually about **groups**: *What's the survival rate in each class? Which state reports the most UFOs?* This notebook also shows how to stack tables and join related tables together.

## ✅ Learning Goals
- Count values with `value_counts()`, including as percentages and count-and-percent tables
- Summarize groups with `groupby()`, by one column or several, and know `.size()` from `.count()`
- Compute several summaries at once with named `.agg()`, and filter the groups afterwards
- Stack tables with `pd.concat()`, and keep track of where the rows came from
- Join related tables with `.merge()`: left vs. inner joins, `suffixes=`, `left_on`/`right_on`
- Check a join with its shape and `validate=`, so it can't silently duplicate rows

> **The core habit:** count categories with `value_counts()`, summarize groups with `groupby()`, stack similar tables with `concat()`, and combine related tables with `merge()`.

## 1. Setup

In [1]:
import pandas as pd

titanic = pd.read_csv('titanic.csv')
titanic.head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


## 2. Counting Values
`value_counts()` counts how often each value appears, most common first. `normalize=True` turns the counts into **proportions**:

In [2]:
titanic['Pclass'].value_counts()

Pclass
3    491
1    216
2    184
Name: count, dtype: int64

In [3]:
titanic['Pclass'].value_counts(normalize=True).round(3)   # share of passengers in each class

Pclass
3    0.551
1    0.242
2    0.207
Name: proportion, dtype: float64

For a report, turn proportions into **percentages** with `.mul(100)`. `value_counts()` sorts by frequency, and `.sort_index()` sorts by the values themselves instead (1st, 2nd, 3rd class). Putting counts and percentages side by side makes a readable summary table:

In [4]:
class_counts = titanic['Pclass'].value_counts().sort_index()
class_percent = titanic['Pclass'].value_counts(normalize=True).sort_index().mul(100).round(1)

pd.DataFrame({'passengers': class_counts, 'percent': class_percent})

,passengers,percent
Pclass,,
1,216,24.2
2,184,20.7
3,491,55.1


Recap from notebook 02: `.unique()` lists the distinct values, `.nunique()` counts them, and `value_counts(dropna=False)` includes missing values in the count.

## 3. Group By: Split → Apply → Combine
`groupby()` does three things in one line:

```text
split    the rows into groups        (one group per class: 1, 2, 3)
apply    a summary to each group     (the mean of Survived)
combine  the answers into a result   (one row per group)
```

Since `Survived` is `1` or `0`, its **mean** is the survival rate:

In [5]:
titanic.groupby('Pclass')['Survived'].mean()

Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64

Read it as: *"group by class, take the `Survived` column, and average it."* The same pattern works with any column and any summary: `.sum()`, `.count()`, `.min()`, `.max()`, `.median()`.

In [6]:
titanic.groupby('Sex')['Fare'].median()

Sex
female    23.0
male      10.5
Name: Fare, dtype: float64

### `.size()` vs. `.count()`
`.size()` counts the **rows** in each group. `.count()` counts the **non-missing values** in one column. They differ whenever that column has gaps:

In [7]:
pd.DataFrame({
    'rows (.size)':          titanic.groupby('Pclass').size(),
    'known ages (.count)':   titanic.groupby('Pclass')['Age'].count(),
})

,rows (.size),known ages (.count)
Pclass,,
1,216,186
2,184,173
3,491,355


### Grouping by More Than One Column
Pass a **list** to get one group per combination:

In [8]:
titanic.groupby(['Pclass', 'Sex'])['Survived'].mean().round(2)

Pclass  Sex   
1       female    0.97
        male      0.37
2       female    0.92
        male      0.16
3       female    0.50
        male      0.14
Name: Survived, dtype: float64

The result has a two-level index. `.reset_index()` turns the group labels back into ordinary columns, which gives you a regular DataFrame that's easier to sort, filter, or save:

In [9]:
survival = titanic.groupby(['Pclass', 'Sex'])['Survived'].mean().round(2).reset_index()
survival.sort_values('Survived', ascending=False)

,Pclass,Sex,Survived
0,1,female,0.97
2,2,female,0.92
4,3,female,0.50
1,1,male,0.37
3,2,male,0.16
5,3,male,0.14


💡 `groupby(..., as_index=False)` gives you regular columns straight away, without needing `.reset_index()`. It's handy for reports:

In [10]:
titanic.groupby('Pclass', as_index=False)['Fare'].mean().round(2)

,Pclass,Fare
0,1,84.15
1,2,20.66
2,3,13.68


## 4. Several Summaries at Once with `.agg()`
`.agg()` computes several summaries in one go. Name each output column with `new_name='function'`:

In [11]:
titanic.groupby('Pclass')['Fare'].agg(
    passengers='count',
    average_fare='mean',
    highest_fare='max',
).round(2)

,passengers,average_fare,highest_fare
Pclass,,,
1,216,84.15,512.33
2,184,20.66,73.50
3,491,13.68,69.55


### Summaries of Different Columns
To summarize **different** columns in one table, group the whole DataFrame and give each output column a `(column, function)` pair:

```python
new_column_name=('existing_column', 'function')
```

In [12]:
class_report = titanic.groupby('Pclass', as_index=False).agg(
    passengers=('PassengerId', 'count'),
    survival_rate=('Survived', 'mean'),
    average_age=('Age', 'mean'),
    average_fare=('Fare', 'mean'),
).round(2)
class_report

,Pclass,passengers,survival_rate,average_age,average_fare
0,1,216,0.63,38.23,84.15
1,2,184,0.47,29.88,20.66
2,3,491,0.24,25.14,13.68


### Row-Level Values First, Then Group
To summarize something that isn't a column yet, **create it first**, then group. Here, family size = siblings/spouses + parents/children + the passenger:

In [13]:
titanic['family_size'] = titanic['SibSp'] + titanic['Parch'] + 1

by_family = titanic.groupby('family_size', as_index=False).agg(
    passengers=('PassengerId', 'count'),
    survival_rate=('Survived', 'mean'),
).round(2)
by_family

,family_size,passengers,survival_rate
0,1,537,0.30
1,2,161,0.55
2,3,102,0.58
3,4,29,0.72
4,5,15,0.20
5,6,22,0.14
6,7,12,0.33
7,8,6,0.00
8,11,7,0.00


### Filtering Groups After Summarizing
Some of those groups are tiny. Six passengers in families of 8 isn't enough to say much. To keep only groups that meet a condition, **summarize first, then filter the summary** like any DataFrame. (In SQL this is `GROUP BY` followed by `HAVING`.)

In [14]:
by_family[by_family['passengers'] >= 20]

,family_size,passengers,survival_rate
0,1,537,0.30
1,2,161,0.55
2,3,102,0.58
3,4,29,0.72
5,6,22,0.14


Traveling alone (30%) or in a large family was risky. Families of 2–4 did best.

## 5. Stacking Tables with `pd.concat()`
When two tables have the **same columns**, like this month's data and last month's, `pd.concat()` stacks them into one. `ignore_index=True` renumbers the rows `0, 1, 2, ...`, so labels aren't repeated:

In [15]:
january = pd.DataFrame({'month': ['Jan', 'Jan'], 'store': ['North', 'South'], 'revenue': [12000, 9500]})
february = pd.DataFrame({'month': ['Feb', 'Feb'], 'store': ['North', 'South'], 'revenue': [14500, 9900]})

sales = pd.concat([january, february], ignore_index=True)
sales

,month,store,revenue
0,Jan,North,12000
1,Jan,South,9500
2,Feb,North,14500
3,Feb,South,9900


In [16]:
sales.groupby('store')['revenue'].sum()

store
North    26500
South    19400
Name: revenue, dtype: int64

💡 **Keep track of where rows came from.** These tables already had a `month` column. If yours don't, add one **before** stacking (`january['month'] = 'Jan'`), or you can't tell the rows apart afterwards.

### `concat` or `merge`?
```text
pd.concat()   stacks MORE ROWS of the same kind of data
              January rows + February rows  →  one longer table

.merge()      adds RELATED COLUMNS through a shared key
              purchases + buyer details     →  wider purchase rows
```

Before combining tables, ask:
- Am I adding **more rows of the same kind** of data? Use `concat`.
- Am I adding **related columns** through a shared key? Use `merge`.
- Which rows must **survive** if there's no match? Choose `'inner'` or `'left'` deliberately.

## 6. Joining Tables with `.merge()`
Real data is often split across tables that share a **key** column. Here, an order table only knows each product's ID, and the price lives in a separate table. `.merge()` lines the rows up by key, much like a SQL `JOIN`:

In [17]:
order_lines = pd.DataFrame({
    'product_id': [101, 102, 101, 105],
    'quantity':   [2, 1, 3, 4],
})
prices = pd.DataFrame({
    'product_id': [101, 102, 103],
    'name':       ['Notebook', 'Backpack', 'Pen'],
    'price':      [4.99, 39.99, 1.50],
})

order_lines.merge(prices, on='product_id', how='left')

,product_id,quantity,name,price
0,101,2,Notebook,4.99
1,102,1,Backpack,39.99
2,101,3,Notebook,4.99
3,105,4,NaN,NaN


Product `105` isn't in the price table, so its `name` and `price` are `NaN`. That's the difference between the two most common joins:

| `how=` | Keeps... | Unmatched rows |
|---|---|---|
| `'left'` | **every** row of the left table | filled with `NaN` |
| `'inner'` | only rows whose key is in **both** tables | dropped |

In [18]:
order_lines.merge(prices, on='product_id', how='inner')

,product_id,quantity,name,price
0,101,2,Notebook,4.99
1,102,1,Backpack,39.99
2,101,3,Notebook,4.99


### A Mini Database: Purchases and Buyers
Real data usually comes as several related tables. Here, each purchase records *who* bought it (`buyer_id`) and where it **shipped** (`state`). Buyer details, including where the buyer **lives** (also called `state`), are in a second table:

In [19]:
purchases = pd.DataFrame({
    'purchase_id': [1001, 1002, 1003, 1004, 1005, 1006],
    'buyer_id':    [1, 2, 1, 3, 4, 2],
    'product_id':  [101, 102, 103, 101, 104, 102],
    'state':       ['IL', 'WI', 'IL', 'MN', 'IL', 'IN'],
    'amount':      [4.99, 39.99, 1.50, 4.99, 24.99, 39.99],
})
buyers = pd.DataFrame({
    'buyer_id':   [1, 2, 3, 4, 5],
    'buyer_name': ['Avery', 'Jordan', 'Riley', 'Morgan', 'Casey'],
    'state':      ['IL', 'WI', 'MN', 'IL', 'IL'],
})

#### Same Column Name in Both Tables: `suffixes=`
Both tables have a `state` column, so pandas can't keep both under one name. By default, it renames them `state_x` and `state_y`, which tells you nothing about which is which. Choose meaningful suffixes:

In [20]:
print(purchases.merge(buyers, on='buyer_id').columns.tolist())

with_buyers = purchases.merge(buyers, on='buyer_id', how='left', suffixes=('_shipped', '_home'))
with_buyers[['purchase_id', 'buyer_name', 'state_shipped', 'state_home', 'amount']]

['purchase_id', 'buyer_id', 'product_id', 'state_x', 'amount', 'buyer_name', 'state_y']


,purchase_id,buyer_name,state_shipped,state_home,amount
0,1001,Avery,IL,IL,4.99
1,1002,Jordan,WI,WI,39.99
2,1003,Avery,IL,IL,1.50
3,1004,Riley,MN,MN,4.99
4,1005,Morgan,IL,IL,24.99
5,1006,Jordan,IN,WI,39.99


Jordan's second order shipped to Indiana, even though Jordan lives in Wisconsin. Without clear suffixes, that would be easy to misread.

#### Keys with Different Names: `left_on` / `right_on`
When the key columns have different names, name each side. Both key columns are kept, so drop the extra one once you've checked the result:

In [21]:
orders = pd.DataFrame({'customer_number': [1, 2, 3], 'total': [19.99, 42.50, 10.00]})

orders.merge(buyers, left_on='customer_number', right_on='buyer_id', how='left').drop(columns=['buyer_id'])

,customer_number,total,buyer_name,state
0,1,19.99,Avery,IL
1,2,42.50,Jordan,WI
2,3,10.00,Riley,MN


#### Check Your Joins: Shape and `validate=`
A merge can quietly **duplicate** rows. If a key appears twice in the right table, every matching row on the left gets copied once for each. Suppose a buyer was accidentally entered twice:

In [22]:
buyers_with_dupe = pd.concat([buyers, pd.DataFrame({'buyer_id': [2], 'buyer_name': ['Jordan B.'], 'state': ['WI']})],
                             ignore_index=True)

merged = purchases.merge(buyers_with_dupe, on='buyer_id', how='left')
print('purchases:', len(purchases), 'rows → after merge:', len(merged), 'rows')

purchases: 6 rows → after merge: 8 rows


Two purchases appeared out of nowhere, and every total computed from `merged` is now wrong. **Always compare row counts before and after a merge.** Better still, tell pandas what relationship you expect with `validate=`, so it stops with an error instead of silently duplicating:

In [23]:
try:
    purchases.merge(buyers_with_dupe, on='buyer_id', how='left', validate='many_to_one')
except pd.errors.MergeError as error:
    print('MergeError:', str(error).splitlines()[0])

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge


| `validate=` | Means |
|---|---|
| `'many_to_one'` | Left keys can repeat (many purchases per buyer), but each right key must be **unique** (one row per buyer) |
| `'one_to_many'` | The reverse |
| `'one_to_one'` | Each key appears once in **both** tables |

For joining details (buyers, products) onto records (purchases), `'many_to_one'` is almost always right. (We catch the error with `try`/`except` here so the notebook keeps running. In your own work, let it stop you!)

### A Real Join: UFO Sightings per Million People
`ufo.csv` has one row per sighting, and `state_abbr.json` maps each state's abbreviation to its population. Count the sightings per US state, then join the populations on:

In [24]:
ufo = pd.read_csv('ufo.csv', usecols=['state', 'country'])
us = ufo[ufo['country'] == 'us']
sightings = us['state'].value_counts().rename_axis('state').reset_index(name='sightings')

# The JSON file is one object of {state: population}, so read it as a Series
population = pd.read_json('state_abbr.json', typ='series').rename('population').rename_axis('state').reset_index()

print(sightings.head(3))
print(population.head(3))

FileNotFoundError: File state_abbr.json does not exist

In [ ]:
joined = sightings.merge(population, on='state', how='left', validate='one_to_one')
print('States with no population:', joined['population'].isna().sum(), 'of', len(joined))

**Every** state failed to match! Look at the keys above: the sightings use `'ca'`, but the populations use `'CA'`. Keys must match **exactly**, so fix one side first:

In [ ]:
sightings['state'] = sightings['state'].str.upper()
joined = sightings.merge(population, on='state', how='left', validate='one_to_one')
print('Still unmatched:', joined[joined['population'].isna()]['state'].tolist())

joined['per_million'] = (joined['sightings'] / joined['population'] * 1_000_000).round(1)
joined.sort_values('per_million', ascending=False).head(5)

Only `PR` (Puerto Rico) is left unmatched, because it isn't in the population file. Dividing by population changes the story: California has the most sightings in total, but it ranks only 17th **per person**.

## 7. A Complete Walkthrough
**Question:** *For each shipping state and product department, how many purchases were there, what was the total revenue, and what was the average purchase?*

The purchases don't record the department, since that lives in a product table. So: **merge** in the detail you need (checking the join), **then** group and summarize:

In [ ]:
products = pd.DataFrame({
    'product_id':   [101, 102, 103, 104, 105],
    'product_name': ['Notebook', 'Backpack', 'Pen', 'Desk Lamp', 'Water Bottle'],
    'department':   ['Office', 'Travel', 'Office', 'Home', 'Travel'],
})

details = purchases.merge(products[['product_id', 'department']],    # only the columns you need
                          on='product_id', how='left', validate='many_to_one')
print('rows before/after:', len(purchases), len(details))

summary = (details
           .groupby(['state', 'department'], as_index=False)
           .agg(purchases=('purchase_id', 'count'),
                total_revenue=('amount', 'sum'),
                average_purchase=('amount', 'mean'))
           .round(2)
           .sort_values(['state', 'total_revenue'], ascending=[True, False]))
summary

Selecting only `product_id` and `department` before merging keeps the result tidy, and avoids clashing column names.

## 🏋️ Practice

### Practice 1 — Where Did They Board? (easy)
Use `value_counts()` on Titanic's `Embarked` column to show the **percentage** of passengers who boarded at each port, including the missing ones.

In [ ]:
import pandas as pd
df = pd.read_csv('titanic.csv')
# Your code here
embarked_pct = df['Embarked'].value_counts(normalize=True, dropna=False) * 100
print(embarked_pct.round(2))

Embarked
S      72.28
C      18.86
Q       8.64
NaN     0.22
Name: proportion, dtype: float64


### Practice 2 — Summarize by Port (medium)
Group Titanic by `Embarked` and use `.agg()` to show, for each port: the number of passengers, the average `Fare`, and the survival rate (the mean of `Survived`). Which port had the highest survival rate?

In [ ]:
# Your code here
port_summary = df.groupby('Embarked').agg(
    passenger_count=('PassengerId', 'count'), # Note: Use 'Survived' if 'PassengerId' isn't in your CSV
    average_fare=('Fare', 'mean'),
    survival_rate=('Survived', 'mean')
)
print(port_summary.round(2))

          passenger_count  average_fare  survival_rate
Embarked                                              
C                     168         59.95           0.55
Q                      77         13.28           0.39
S                     644         27.08           0.34


### Practice 3 — Join, Then Group (harder)
Build these two tables, merge them on `student_id`, and then find the **average score for each major**. Which join (`'left'` or `'inner'`) did you use, and what happened to student `5`?

In [ ]:
scores = pd.DataFrame({'student_id': [1, 2, 3, 4, 5], 'score': [88, 92, 75, 81, 95]})
majors = pd.DataFrame({'student_id': [1, 2, 3, 4], 'major': ['CS', 'Math', 'CS', 'Biology']})
# Your code here
merged = pd.merge(scores, majors, on='student_id', how='inner')
avg_by_major = merged.groupby('major')['score'].mean()

print("--- Merged Table (Inner) ---")
print(merged)
print("\n--- Average Score by Major ---")
print(avg_by_major)

--- Merged Table (Inner) ---
   student_id  score    major
0           1     88       CS
1           2     92     Math
2           3     75       CS
3           4     81  Biology

--- Average Score by Major ---
major
Biology    81.0
CS         81.5
Math       92.0
Name: score, dtype: float64


### Practice 4 — Only the Big Groups (medium)
Group Titanic by **both** `Embarked` and `Pclass`. Use `.agg()` with `(column, function)` pairs to get the number of passengers and the survival rate for each combination. Then keep only the groups with **at least 50** passengers, sorted by survival rate from highest to lowest.

In [ ]:
# Your code here
grouped = df.groupby(['Embarked', 'Pclass']).agg(
    passenger_count=('Survived', 'count'),
    survival_rate=('Survived', 'mean')
)
big_groups = grouped[grouped['passenger_count'] >= 50].sort_values(by='survival_rate', ascending=False)
print(big_groups.round(3))

                 passenger_count  survival_rate
Embarked Pclass                                
C        1                    85          0.694
S        1                   127          0.583
         2                   164          0.463
C        3                    66          0.379
Q        3                    72          0.375
S        3                   353          0.190


### Practice 5 — Join Carefully (harder)
Using `purchases` and `buyers` from section 6:
1. Merge them with `how='inner'`, `suffixes=('_shipped', '_home')`, and `validate='many_to_one'`, and compare the row count to `purchases`.
2. Which buyer never appears in the result, and why? Would a **left** join from `buyers` to `purchases` show them?
3. Find each buyer's total spending, using their `buyer_name`.

In [ ]:
# Your code here
purchases = pd.DataFrame({
    'purchase_id': [1001, 1002, 1003, 1004, 1005, 1006],
    'buyer_id':    [1, 2, 1, 3, 4, 2],
    'product_id':  [101, 102, 103, 101, 104, 102],
    'state':       ['IL', 'WI', 'IL', 'MN', 'IL', 'IN'],
    'amount':      [4.99, 39.99, 1.50, 4.99, 24.99, 39.99],
})
buyers = pd.DataFrame({
    'buyer_id':   [1, 2, 3, 4, 5],
    'buyer_name': ['Avery', 'Jordan', 'Riley', 'Morgan', 'Casey'],
    'state':      ['IL', 'WI', 'MN', 'IL', 'IL'],
})

merged = pd.merge(
    purchases,
    buyers,
    on='buyer_id',
    how='inner',
    suffixes=('_shipped', '_home'),
    validate='many_to_one'
)

print(f"Purchases rows: {len(purchases)}")
print(f"Merged rows:    {len(merged)}")
print("\n--- Merged Table ---")
print(merged)

missing_buyers = buyers[~buyers['buyer_id'].isin(merged['buyer_id'])]
print("\n--- Missing Buyer ---")
print(missing_buyers)

left_joined = pd.merge(
    buyers,
    purchases,
    on='buyer_id',
    how='left',
    suffixes=('_home', '_shipped')
)
print("\n--- Left Join (buyers -> purchases) ---")
print(left_joined)

total_spending = merged.groupby('buyer_name')['amount'].sum()
print("\n--- Total Spending by Buyer ---")
print(total_spending)


Purchases rows: 6
Merged rows:    6

--- Merged Table ---
   purchase_id  buyer_id  product_id state_shipped  amount buyer_name  \
0         1001         1         101            IL    4.99      Avery   
1         1002         2         102            WI   39.99     Jordan   
2         1003         1         103            IL    1.50      Avery   
3         1004         3         101            MN    4.99      Riley   
4         1005         4         104            IL   24.99     Morgan   
5         1006         2         102            IN   39.99     Jordan   

  state_home  
0         IL  
1         WI  
2         IL  
3         MN  
4         IL  
5         WI  

--- Missing Buyer ---
   buyer_id buyer_name state
4         5      Casey    IL

--- Left Join (buyers -> purchases) ---
   buyer_id buyer_name state_home  purchase_id  product_id state_shipped  \
0         1      Avery         IL       1001.0       101.0            IL   
1         1      Avery         IL       1003.0     

## ✏️ Your Turn
1. Turn this list of purchases into a DataFrame. Then use `groupby` to find each buyer's **total** and **number** of purchases.
2. Add a `month` column, then stack two new purchases from a second DataFrame (with their own `month`) using `pd.concat()`, and recompute the totals per buyer **and** per month.
3. Use `value_counts()` to find the most frequent buyer, and show each buyer's share of purchases as a **percentage**.
4. In a comment, explain why `pd.concat()` fits monthly files, while `.merge()` fits adding buyer details to purchases.

In [ ]:
purchases = pd.DataFrame([
    {'purchase_id': 1, 'buyer': 'Avery',  'amount': 25.00},
    {'purchase_id': 2, 'buyer': 'Jordan', 'amount': 42.50},
    {'purchase_id': 3, 'buyer': 'Avery',  'amount': 18.75},
    {'purchase_id': 4, 'buyer': 'Sam',    'amount': 9.99},
])
# Your turn here
buyer_summary = purchases.groupby('buyer').agg(
    total_spent=('amount', 'sum'),
    purchase_count=('purchase_id', 'count')
)
print("--- 1. Total and Number of Purchases per Buyer ---")
print(buyer_summary)

purchases['month'] = 'January'

new_purchases = pd.DataFrame([
    {'purchase_id': 5, 'buyer': 'Jordan', 'amount': 30.00, 'month': 'February'},
    {'purchase_id': 6, 'buyer': 'Avery',  'amount': 15.25, 'month': 'February'},
])

all_purchases = pd.concat([purchases, new_purchases], ignore_index=True)

monthly_summary = all_purchases.groupby(['buyer', 'month']).agg(
    total_spent=('amount', 'sum'),
    purchase_count=('purchase_id', 'count')
)
print("\n--- 2. Totals per Buyer and Month ---")
print(monthly_summary)

buyer_pct = all_purchases['buyer'].value_counts(normalize=True) * 100
most_frequent = buyer_pct.idxmax()

print("\n--- 3. Share of Purchases (%) ---")
print(buyer_pct.round(2))
print(f"Most frequent buyer: {most_frequent} ({buyer_pct.max():.2f}%)")


# 4. Explanation:
# pd.concat() stacks tables vertically (adding rows) when datasets share the same columns,
# which fits combining monthly files that record the same types of transactions.
# .merge() joins tables horizontally (adding columns) by matching a shared key column,
# which fits looking up and attaching buyer profile details (like address or email) to each purchase.

--- 1. Total and Number of Purchases per Buyer ---
        total_spent  purchase_count
buyer                              
Avery         43.75               2
Jordan        42.50               1
Sam            9.99               1

--- 2. Totals per Buyer and Month ---
                 total_spent  purchase_count
buyer  month                                
Avery  February        15.25               1
       January         43.75               2
Jordan February        30.00               1
       January         42.50               1
Sam    January          9.99               1

--- 3. Share of Purchases (%) ---
buyer
Avery     50.00
Jordan    33.33
Sam       16.67
Name: proportion, dtype: float64
Most frequent buyer: Avery (50.00%)


## 🔥 Challenge (Optional) — The Peak Year for UFO Sightings
Load the `date_time` column of `ufo.csv`, convert it with `pd.to_datetime(..., errors='coerce')`, and pull out the year with `.dt.year`. Then use `value_counts()` (or `groupby`) to find the year with the most sightings. *Hint:* a few hundred dates fail to convert. Why might that be? Look at some of the original text for the rows that became `NaT`.

In [ ]:
# Your investigation here


---
### 📝 Summary
| Task | Code |
|---|---|
| Count values | `df['col'].value_counts()`, add `normalize=True` for proportions, `.mul(100)` for percent |
| Summarize by group | `df.groupby('col')['other'].mean()` |
| Rows vs. values per group | `.size()` counts rows; `.count()` counts non-missing values |
| Group by several columns | `df.groupby(['a', 'b'], as_index=False)['other'].sum()` |
| Several summaries | `df.groupby('col').agg(total=('amount', 'sum'), average=('amount', 'mean'))` |
| Filter groups | Summarize first, then `summary[summary['count'] >= n]` |
| Stack tables | `pd.concat([df1, df2], ignore_index=True)` |
| Join tables | `left.merge(right, on='key', how='left')` |
| Same column names | `suffixes=('_left', '_right')` |
| Different key names | `left_on='a', right_on='b'` |
| Check a join | Compare `len()` before and after, and use `validate='many_to_one'` |

Next up: the **Pandas Wrangle Lab**, which puts the whole module together.